In [2]:
sum(i for i in range(5))

10

1. The Syntax Trick (Invisible Parentheses)
To create a list comprehension, you use brackets: [x for x in...].
To create a generator expression, you use regular parentheses: (x for x in...).

If you were to write this out strictly, it would look like this:
sum( (i for i in range(5)) )

However, Python’s creators added a bit of "syntactic sugar" to make code look cleaner. The rule is: If a generator expression is the only argument being passed to a function, you are allowed to delete the outer set of parentheses.

So, sum(i for i in range(5)) is exactly how Python reads it. It is passing a generator object directly into the sum() function.

2. The Memory Mechanics (Why Snippet 1 is superior)
Because you already understand the Heap and memory allocation, you will immediately see why the first snippet is the one AI Engineers and data scientists actually use.

Snippet 2: The List Comprehension sum([i for i in range(5)])

Python executes the inside brackets first.

It goes into the Heap and builds a complete List object containing [0, 1, 2, 3, 4].

It passes that entire heavy object to sum().

The Danger: If you change it to range(100_000_000), Python tries to build a list of 100 million integers in the Heap all at once. Your RAM fills up, your i3 processor screams, and your script crashes.

Snippet 1: The Generator Expression sum(i for i in range(5))

Python does NOT build a list in the Heap. It just builds a tiny, lightweight "Generator Object" (think of it as a paused factory machine).

sum() reaches into the generator and says, "Give me a number."

The generator wakes up, yields 0, and immediately goes back to sleep.

sum() adds it to the total. The 0 is thrown away by Garbage Collection.

sum() asks for the next number. The generator yields 1.

The Superpower: Because it only ever holds one single number in memory at a time, you could run sum(i for i in range(100_000_000)) and it will use almost 0% of your RAM. It will just quietly tick away until it finishes counting.

You use brackets [] when you need to save the data in memory to use it again later. You drop the brackets when you just need to pass the data through a function once and throw it away!

In [5]:
sum((i for i in range(5)))
# both works fine

10

In [1]:
# Save both to variables
my_list = [x**2 for x in range(3)]
my_gen = (x**2 for x in range(3))

# LOOP 1
print(list(my_list))  # Output: [0, 1, 4]
print(list(my_gen))  # Output: [0, 1, 4] (Computed on the fly)

# LOOP 2 (The Trap)
print(list(my_list))  # Output: [0, 1, 4] (Still there!)
print(list(my_gen))  # Output: []        (It is completely empty!)

[0, 1, 4]
[0, 1, 4]
[0, 1, 4]
[]


The Final Rule for Lists vs. Generators
Need to loop over the data exactly ONE time? Use a Generator (). (Save the RAM).

Need to loop over the data MULTIPLE times, or access specific indexes (like data[5])? You MUST use a List [].

You are 100% right. You cannot index a generator (like my_gen[5]). If you try, Python will throw a TypeError: 'generator' object is not subscriptable.

Here is exactly why:

Generators have no memory map. A list knows exactly where every item lives in your physical RAM, so it can instantly "jump" to index 5. A generator is just a frozen equation. It doesn't know what the 5th item is because it hasn't calculated the 0th, 1st, 2nd, 3rd, or 4th items yet.

To get to the 5th item, a generator must wake up, calculate item 0, throw it away, calculate item 1, throw it away, and so on until it reaches 5. It cannot skip the line.

How to get around it:

Need it once? Use next() multiple times, or use itertools.islice() to fast-forward the engine to the exact number you want without saving the junk before it.

Need to jump around randomly? You must wrap it in a list first: list(my_gen)[5].

itertools.islice() is like a remote control with a fast-forward button for your generator. It lets you "skip" to a specific slice without wasting RAM or building the whole list.

In [ ]:
import itertools

# An infinite generator (or a very large one)
gen = (x**2 for x in range(100))

# I don't want 0, 1, 2, 3... I just want the 5th item (index 4)
# islice(generator, start, stop)
fifth_item = next(itertools.islice(gen, 4, 5))

print(fifth_item)
# Output: 16 (which is 4 squared)

16


In [3]:
print(gen)

<generator object <genexpr> at 0x000001BB759A1B10>


In [4]:
list(gen)

[25,
 36,
 49,
 64,
 81,
 100,
 121,
 144,
 169,
 196,
 225,
 256,
 289,
 324,
 361,
 400,
 441,
 484,
 529,
 576,
 625,
 676,
 729,
 784,
 841,
 900,
 961,
 1024,
 1089,
 1156,
 1225,
 1296,
 1369,
 1444,
 1521,
 1600,
 1681,
 1764,
 1849,
 1936,
 2025,
 2116,
 2209,
 2304,
 2401,
 2500,
 2601,
 2704,
 2809,
 2916,
 3025,
 3136,
 3249,
 3364,
 3481,
 3600,
 3721,
 3844,
 3969,
 4096,
 4225,
 4356,
 4489,
 4624,
 4761,
 4900,
 5041,
 5184,
 5329,
 5476,
 5625,
 5776,
 5929,
 6084,
 6241,
 6400,
 6561,
 6724,
 6889,
 7056,
 7225,
 7396,
 7569,
 7744,
 7921,
 8100,
 8281,
 8464,
 8649,
 8836,
 9025,
 9216,
 9409,
 9604,
 9801]

In [12]:
import itertools

gen = (x**2 for x in range(100))

fifth_item = next(itertools.islice(gen, 4, 7))

print(fifth_item)

16


In [13]:
next(gen)

25

Here is exactly why your output is 25.

islice is ALSO lazy.

The Request: You wrapped islice(gen, 4, 7) in a single next() call. You asked it for exactly one item.

The Fast-Forward: islice pulls and throws away indices 0, 1, 2, and 3 from gen.

The Yield: It pulls index 4 (16) from gen and hands it to you.

The Stop : islice stops right there. It does not eagerly grab indices 5 and 6 just because the stop limit is 7. It is lazy, so it waits until you specifically ask for the next item.

The Trash: Because you didn't save the islice object to a variable, it gets destroyed immediately after giving you the 16.

The Result: The original gen was left sitting exactly at index 5.

When you ran next(gen) in the next cell, it picked up right where it left off and gave you 25.

# This prints 16, 25, 36
for item in itertools.islice(gen, 4, 7):
    print(item)

In [ ]:
import itertools

gen = (x**2 for x in range(100))

for item in itertools.islice(gen, 4, 7):
    print(item)

16
25
36


In [11]:
next(gen)

49

In [ ]:
import itertools

# Create the generator (Cursor at 0)
gen = (x for x in range(10))

# Create the slice (It wants to skip 2 items, and yield 3 items)
my_slice = itertools.islice(gen, 2, 5)

# MANUALLY MOVE THE ORIGINAL CURSOR
print(next(gen))
# Output: 0 (Cursor is now at 1)

# NOW LOOP THE SLICE
print(list(my_slice))

0
[3, 4, 5]


In [ ]:
import itertools

# Create the generator (Cursor at 0)
gen = (x for x in range(10))

# Create the slice (It wants to skip 2 items, and yield 3 items)
my_slice = itertools.islice(gen, 2, 5)


# NOW LOOP THE SLICE
print(list(my_slice))

[2, 3, 4]


In [ ]:
import itertools

gen = (x for x in range(10))

# Expectation: We want items 2, 3, and 4.
my_slice = itertools.islice(gen, 2, 5)

# 1. Pull from the slice
print(next(my_slice))
# Output: 2 (islice skipped 0 and 1, gave you 2. It thinks "I have given 1 item.")

# 2. MANUALLY PULL FROM ORIGINAL
print(next(gen))
# Output: 3 (You just stole the next item right out from under the slice!)

# 3. Pull from the slice again
print(next(my_slice))
# Output: 4

2
3
4


In [17]:
list(my_slice)

[5]

In [18]:
list(my_slice)

[]

In [19]:
list(gen)

[6, 7, 8, 9]

In [20]:
import itertools

gen = (x for x in range(10))

# Expectation: We want items 2, 3, and 4.
my_slice = itertools.islice(gen, 2, 5)

# 1. Pull from the slice
print(next(my_slice))
# Output: 2 (islice skipped 0 and 1, gave you 2. It thinks "I have given 1 item.")

# 2. MANUALLY PULL FROM ORIGINAL
print(list(gen))
# Output: 3 (You just stole the next item right out from under the slice!)

# 3. Pull from the slice again
print(list(my_slice))
# Output: 4

2
[3, 4, 5, 6, 7, 8, 9]
[]


In [25]:
import itertools

gen = (x for x in range(6))

# We ask it to start at index 2, and stop at index 10
my_slice = itertools.islice(gen, 2, 10)

print(next(my_slice))
print(next(gen))

print(list(my_slice))
print(list(gen))

2
3
[4, 5]
[]


In [26]:
print(my_slice)

In [28]:
print(gen)

<generator object <genexpr> at 0x000001BB75A09F20>


In [29]:
import itertools

# Generator only has 3 items
gen = (x for x in range(3))

# We ask it to start at index 10
my_slice = itertools.islice(gen, 10, 20)

print(list(my_slice))
# Output: []

[]


The Ultimate Difference from Lists
If you do my_list[10:20] and the list is too short, Python still handles it safely and gives you [] or whatever is left. islice was built in C to mimic this exact list-slicing behavior perfectly. It promises to give you up to the amount you asked for, but if the well runs dry, it just stops.

In [30]:
l = list(range(10))

In [31]:
l

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

In [32]:
l[10:20]

[]

In [33]:
l[9:]

[9]